In [ ]:
import os
if not os.path.exists("gaussian-splatting"):
    !git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive
%cd gaussian-splatting
OUTPUT_DIR = "/content/gaussian-splatting/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
%%writefile utils/vgg_loss.py
import torch, torch.nn as nn, torchvision
class VGGPerceptualLoss(nn.Module):
    def __init__(self, device="cuda"):
        super().__init__()
        vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features[:16].to(device).eval()
        for p in self.features.parameters(): p.requires_grad = False
        self.layers = {"relu1_2": 3, "relu2_2": 8, "relu3_3": 15}
        self.layer_weights = {"relu1_2": 1.0/2.6, "relu2_2": 1.0/4.8, "relu3_3": 1.0/3.7}
        self.mean = torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
        self.std = torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
    def forward(self, pred, target):
        if pred.dim() == 3: pred, target = pred.unsqueeze(0), target.unsqueeze(0)
        p, t = (pred-self.mean)/self.std, (target-self.mean)/self.std
        loss = 0.0
        for name, idx in self.layers.items():
            start = 0 if name=="relu1_2" else (4 if name=="relu2_2" else 9)
            for i in range(start, idx+1):
                p, t = self.features[i](p), self.features[i](t)
            loss += self.layer_weights[name] * nn.functional.l1_loss(p, t)
        return loss
print("VGG loss created")


In [ ]:
import urllib.request, zipfile, os
d = "/content/data"
os.makedirs(d, exist_ok=True)
if not os.path.exists(f"{d}/tandt"):
    print("Downloading...")
    urllib.request.urlretrieve("https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip", f"{d}/tandt_db.zip")
    with zipfile.ZipFile(f"{d}/tandt_db.zip") as z: z.extractall(d)
    print("Done")
else: print("Already exists")


In [ ]:
!python train.py \
  -s /content/data/tandt/truck \
  -m /content/gaussian-splatting/output/truck_improved \
  --lambda_vgg 0.2 --iterations 7000 \
  --test_iterations 7000 --save_iterations 7000 \
  --quiet --disable_viewer
print("\nImproved training done")


In [ ]:
import matplotlib.pyplot as plt, numpy as np, os
from PIL import Image
def load(mp, split="test", it=7000):
    rd = f"{mp}/{split}/ours_{it}/renders"
    gd = f"{mp}/{split}/ours_{it}/gt"
    imgs = sorted(os.listdir(rd))
    if not imgs: return None, None
    r = np.array(Image.open(f"{rd}/{imgs[0]}"))
    g = np.array(Image.open(f"{gd}/{imgs[0]}"))
    return r, g

base = "/content/gaussian-splatting/output"
or_, og_ = load(f"{base}/truck_original")
ir_, ig_ = load(f"{base}/truck_improved")

if or_ is not None:
    fig, ax = plt.subplots(2,3,figsize=(15,10))
    ax[0,0].imshow(og_); ax[0,0].set_title("GT"); ax[0,0].axis("off")
    ax[0,1].imshow(or_); ax[0,1].set_title("Original"); ax[0,1].axis("off")
    d1 = np.abs(or_.astype(float)-og_.astype(float)).mean(2)
    a=ax[0,2].imshow(d1,cmap="hot"); ax[0,2].set_title(f"Err {d1.mean():.1f}"); ax[0,2].axis("off"); plt.colorbar(a,ax=ax[0,2],fraction=0.046)
    ax[1,0].imshow(ig_); ax[1,0].set_title("GT"); ax[1,0].axis("off")
    ax[1,1].imshow(ir_); ax[1,1].set_title("Improved (+VGG)"); ax[1,1].axis("off")
    d2 = np.abs(ir_.astype(float)-ig_.astype(float)).mean(2)
    a=ax[1,2].imshow(d2,cmap="hot"); ax[1,2].set_title(f"Err {d2.mean():.1f}"); ax[1,2].axis("off"); plt.colorbar(a,ax=ax[1,2],fraction=0.046)
    plt.tight_layout()
    plt.savefig(f"{base}/comparison.png",dpi=150,bbox_inches="tight")
    plt.show()
    print("\nSaved comparison.png")
else: print("No renders yet")
